In [ ]:
import sys
from pathlib import Path
import datetime as dt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.py_eddy_tracker.observations.tracking import TrackEddiesObservations

EXPERIMENT = "gulf_stream_20241001_20250701"
DIAGNOSTIC_PIGMENTS = ["Fuco", "HexFuco", "Perid", "Zea", "DV chla", "Allo", "MV chlb"]
PFT_COLS = ["Diatoms", "Dinoflagellates", "Haptophytes", "Cryptophytes", "Green_algae", "Cyanobacteria"]

In [ ]:
frames = []
for polarity in ("cyclone", "anticyclone"):
    pigments_dir = ROOT / "outputs" / EXPERIMENT / "pigments" / polarity
    for fp in sorted(pigments_dir.glob("eddy_*_pigments.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        frames.append(df)

pigments = pd.concat(frames, ignore_index=True)
pigment_cols = [c for c in pigments.columns
                if c not in ["track_id", "date", "pixel_lon", "pixel_lat",
                             "center_lon", "center_lat", "coverage", "polarity"]]
print(
    f"pigment_pixels: {len(pigments)}\n"
    f"pigment_tracks_by_polarity: {pigments.groupby('polarity')['track_id'].nunique().to_dict()}"
)

In [ ]:
pft_frames = []
for polarity in ("cyclone", "anticyclone"):
    pft_dir = ROOT / "outputs" / EXPERIMENT / "pft" / polarity
    for fp in sorted(pft_dir.glob("eddy_*_pfts.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"] = polarity
        pft_frames.append(df)

pfts = pd.concat(pft_frames, ignore_index=True)
print(
    f"pft_pixels: {len(pfts)}\n"
    f"pft_tracks_by_polarity: {pfts.groupby('polarity')['track_id'].nunique().to_dict()}"
)

In [ ]:
def load_track_props(polarity):
    PET_EPOCH = dt.date(1950, 1, 1)
    zarr_path = ROOT / "outputs" / EXPERIMENT / "eddy_track" / polarity / f"{polarity}_tracks.zarr"
    tracked = TrackEddiesObservations.load_file(str(zarr_path))

    unique_ids = np.unique(tracked.track)
    lifetime = {}
    for tid in unique_ids:
        t = tracked.time[tracked.track == tid]
        lifetime[tid] = int(t.max() - t.min()) + 1

    dates = [PET_EPOCH + dt.timedelta(days=int(d)) for d in tracked.time]
    return pd.DataFrame({
        "track_id": tracked.track.astype(int),
        "date": pd.to_datetime(dates),
        "polarity": polarity,
        "center_lon": (tracked.longitude + 180) % 360 - 180,
        "center_lat": tracked.latitude,
        "radius_km": tracked.radius_e / 1000,
        "amplitude_m": tracked.amplitude,
        "speed_avg": tracked.speed_average,
        "lifetime_days": [lifetime[tid] for tid in tracked.track],
    })

track_props = pd.concat(
    [load_track_props(p) for p in ("cyclone", "anticyclone")],
    ignore_index=True,
)

In [ ]:
pigments = pigments.merge(
    track_props[["track_id", "date", "polarity", "radius_km", "amplitude_m", "lifetime_days"]],
    on=["track_id", "date", "polarity"],
    how="left",
)

# Per-eddy-date spatial medians
pig_medians = (
    pigments.groupby(["track_id", "date", "polarity"])[pigment_cols]
    .median()
    .reset_index()
)

pft_medians = (
    pfts.groupby(["track_id", "date", "polarity"])[PFT_COLS]
    .median()
    .reset_index()
)

# Filter retrieval failures
pig_medians = pig_medians[pig_medians["T chla"] >= 0.01].copy()
pft_medians = pft_medians[pft_medians[PFT_COLS].sum(axis=1) > 0].copy()

SEASON_MAP = {10: "Fall", 11: "Fall", 12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring", 4: "Spring", 5: "Spring", 6: "Summer"}
SEASON_ORDER = ["Fall", "Winter", "Spring", "Summer"]

for df in (pig_medians, pft_medians):
    df["month"] = df["date"].dt.month
    df["season"] = df["month"].map(SEASON_MAP)

print(f"filtered_pigment_eddy_dates: {len(pig_medians)}")
print(f"filtered_pft_eddy_dates: {len(pft_medians)}")

## Pooled seasonal overview

In [ ]:
MONTH_ORDER = [10, 11, 12, 1, 2, 3, 4, 5, 6]
MONTH_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"]

# Only count pixels from eddy-dates that passed the T chla filter
valid_keys = pig_medians[["track_id", "date", "polarity"]]
pig_filtered = pigments.merge(valid_keys, on=["track_id", "date", "polarity"])
pig_filtered["month"] = pig_filtered["date"].dt.month

sample_sizes = []
for m in MONTH_ORDER:
    sub = pig_medians[pig_medians["month"] == m]
    px = pig_filtered[pig_filtered["month"] == m]
    sample_sizes.append({
        "month": MONTH_LABELS[MONTH_ORDER.index(m)],
        "n_eddies": sub["track_id"].nunique(),
        "n_eddy_dates": len(sub),
        "n_pixels": len(px),
    })

sample_df = pd.DataFrame(sample_sizes)
sample_df

Monthly T chla split by polarity, with one value per eddy per month defined as the median across that eddy's valid observation dates in that month. 
Faint gray points show individual eddies; colored points and lines show the monthly median across eddies, with 95% bootstrap confidence intervals. 
This makes the cyclone-anticyclone seasonal contrast visible without overweighting eddies that were sampled more often within a month.

In [ ]:
eddy_monthly = (
    pig_medians.groupby(["track_id", "polarity", "month"])["T chla"]
    .median()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
xpos = np.arange(1, len(MONTH_ORDER) + 1)

for ax, polarity, color in zip(axes, ["cyclone", "anticyclone"], ["tab:blue", "tab:red"]):
    summary_rows = []
    rng = np.random.default_rng(42)
    jitter_rng = np.random.default_rng(42)

    for x, m, label in zip(xpos, MONTH_ORDER, MONTH_LABELS):
        vals = eddy_monthly.loc[(eddy_monthly["polarity"] == polarity) & (eddy_monthly["month"] == m), "T chla"].to_numpy()

        if len(vals) == 0:
            summary_rows.append({
                "month": m,
                "label": label,
                "median_tchla": np.nan,
                "ci_lo": np.nan,
                "ci_hi": np.nan,
                "n_eddies": 0,
            })
            continue

        boot = []
        for _ in range(2000):
            sample = rng.choice(vals, size=len(vals), replace=True)
            boot.append(np.median(sample))
        lo, hi = np.percentile(boot, [2.5, 97.5])
        summary_rows.append({
            "month": m,
            "label": label,
            "median_tchla": np.median(vals),
            "ci_lo": lo,
            "ci_hi": hi,
            "n_eddies": len(vals),
        })

        x_jitter = x + jitter_rng.uniform(-0.12, 0.12, size=len(vals))
        ax.scatter(x_jitter, vals, s=18, color="0.7", alpha=0.45, linewidths=0, zorder=1)

    summary_df = pd.DataFrame(summary_rows)

    ax.errorbar(
        xpos,
        summary_df["median_tchla"],
        yerr=[
            summary_df["median_tchla"] - summary_df["ci_lo"],
            summary_df["ci_hi"] - summary_df["median_tchla"],
        ],
        fmt="o-",
        color=color,
        lw=2,
        ms=6,
        capsize=4,
        zorder=3,
    )

    ax.set_xticks(xpos)
    ax.set_xticklabels(MONTH_LABELS)
    ax.set_xlabel("Month")
    ax.set_title(polarity.capitalize())
    ax.grid(axis="y", alpha=0.2)

axes[0].set_ylabel("T chla")
fig.suptitle("Monthly T chla by polarity", y=1.03)
fig.tight_layout()


Seasonal community composition expressed as mean PFT fraction of total Chla, averaged across all eddy-date observations per season. Each bar sums to 1. Seasons defined as Fall (Oct–Nov), Winter (Dec–Feb), Spring (Mar–May), Summer (Jun).

In [ ]:
seasonal_pft = pft_medians.groupby("season")[PFT_COLS].mean()
seasonal_pft = seasonal_pft.loc[SEASON_ORDER]

seasonal_frac = seasonal_pft.div(seasonal_pft.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
seasonal_frac.plot(kind="bar", stacked=True, ax=ax, width=0.6)
ax.set_ylabel("PFT fraction")
ax.set_title("Community composition by season")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xticklabels(SEASON_ORDER, rotation=0)
fig.tight_layout()

## Polarity-split seasonality

Monthly T chla distributions split by eddy polarity (cyclonic left, anticyclonic right), with shared y-axis for direct comparison. Each box summarizes per-eddy spatial median T chla for that month and polarity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, (pol, color) in zip(axes, [("cyclone", "tab:blue"), ("anticyclone", "tab:red")]):
    sub = pig_medians[pig_medians["polarity"] == pol]
    groups = [sub[sub["month"] == m]["T chla"].values for m in MONTH_ORDER]
    bp = ax.boxplot(groups, labels=MONTH_LABELS, patch_artist=True,
                    medianprops=dict(color="black"))
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    ax.set_xlabel("Month")
    ax.set_title(pol.capitalize())

axes[0].set_ylabel("T chla")
fig.suptitle("Monthly T chla by polarity", y=1.02)
fig.tight_layout()

Monthly median diagnostic-pigment concentration by polarity, with IQR shading. Each point is the monthly median across per-eddy-date spatial means; shaded bands span the 25th–75th percentile. Blue = cyclonic, red = anticyclonic.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

pol_colors = {"cyclone": "tab:blue", "anticyclone": "tab:red"}

for i, pig in enumerate(DIAGNOSTIC_PIGMENTS):
    ax = axes.flat[i]
    for pol in ("cyclone", "anticyclone"):
        sub = pig_medians[pig_medians["polarity"] == pol]
        monthly = sub.groupby("month")[pig]
        medians = monthly.median().reindex(MONTH_ORDER)
        q25 = monthly.quantile(0.25).reindex(MONTH_ORDER)
        q75 = monthly.quantile(0.75).reindex(MONTH_ORDER)

        x = range(len(MONTH_ORDER))
        ax.plot(x, medians.values, "o-", color=pol_colors[pol], label=pol, ms=4)
        ax.fill_between(x, q25.values, q75.values, color=pol_colors[pol], alpha=0.15)

    ax.set_xticks(range(len(MONTH_ORDER)))
    ax.set_xticklabels(MONTH_LABELS, fontsize=8)
    ax.set_title(pig)

axes.flat[0].legend()
axes.flat[-1].set_visible(False)
fig.suptitle("Diagnostic pigments by month and polarity (median + IQR)", y=1.01)
fig.tight_layout()

PFT community composition by season and polarity. Stacked bars show mean PFT fractions normalized to 1 within each season–polarity group. Comparing cyclonic and anticyclonic bars within each season reveals polarity-dependent shifts in community structure.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)

for ax, season in zip(axes, SEASON_ORDER):
    for j, pol in enumerate(("cyclone", "anticyclone")):
        sub = pft_medians[(pft_medians["season"] == season) & (pft_medians["polarity"] == pol)]
        means = sub[PFT_COLS].mean()
        fracs = means / means.sum()
        bottom = 0
        for k, pft in enumerate(PFT_COLS):
            ax.bar(j, fracs[pft], bottom=bottom, color=f"C{k}",
                   label=pft if (season == "Fall" and j == 0) else None)
            bottom += fracs[pft]

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Cyc", "Anti"], fontsize=9)
    ax.set_title(season)

axes[0].set_ylabel("PFT fraction")
axes[0].legend(bbox_to_anchor=(-0.3, 1), loc="upper right", fontsize=8)
fig.suptitle("PFT composition by season and polarity", y=1.02)
fig.tight_layout()

## Per-eddy trajectories

In [ ]:
pooled_monthly = pig_medians.groupby("month")["T chla"].median()

Per-eddy T chla trajectories over calendar time. Each thin line traces one eddy's spatial median T chla; the dashed black line is the pooled monthly median across all eddies. Only eddy-dates with T chla ≥ 0.01 mg/m³ are included. Left: cyclonic; right: anticyclonic.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, (pol, color) in zip(axes, [("cyclone", "tab:blue"), ("anticyclone", "tab:red")]):
    sub = pig_medians[pig_medians["polarity"] == pol]
    for _, grp in sub.sort_values("date").groupby("track_id"):
        ax.plot(grp["date"], grp["T chla"], color=color, alpha=0.25, lw=0.8)

    ref_dates = [dt.date(2025 if m <= 6 else 2024, m, 15) for m in MONTH_ORDER]
    ax.plot(ref_dates, pooled_monthly.reindex(MONTH_ORDER).values,
            "k--", lw=2, label="pooled monthly median")

    ax.set_title(f"{pol.capitalize()} (n={sub['track_id'].nunique()})")
    ax.set_xlabel("Date")
    ax.set_ylim(0, 2)
    ax.legend(fontsize=8)

axes[0].set_ylabel("T chla (spatial median)")
fig.suptitle("Eddy T chla over calendar time", y=1.02)
fig.autofmt_xdate()
fig.tight_layout()

In [ ]:
# Eddies spanning >= 3 calendar months with >= 8 PACE observations
candidates = pig_medians.groupby(["track_id", "polarity"]).agg(
    n_obs=("date", "size"),
    first=("date", "min"),
    last=("date", "max"),
).reset_index()
candidates["months_spanned"] = (
    (candidates["last"].dt.year - candidates["first"].dt.year) * 12
    + candidates["last"].dt.month - candidates["first"].dt.month + 1
)
candidates = candidates[(candidates["n_obs"] >= 8) & (candidates["months_spanned"] >= 3)]
candidates = candidates.sort_values("months_spanned", ascending=False)

for pol in ("cyclone", "anticyclone"):
    sub = candidates[candidates["polarity"] == pol]
    print(
        f"polarity: {pol}\n"
        f"candidates: {len(sub)}"
    )
    print(sub[["track_id", "n_obs", "first", "last", "months_spanned"]].head(8).to_string(index=False))
    print()

Case study eddies selected for long temporal coverage (≥ 3 calendar months, ≥ 8 PACE observations). Left column: cyclonic; right column: anticyclonic. Solid black line shows T chla (left y-axis); dashed colored lines show the three dominant PFTs by mean concentration (right y-axis).

In [ ]:
CASE_CYCLONES = [3, 27, 33]
CASE_ANTICYCLONES = [1, 18, 24]

case_ids = {
    "cyclone": CASE_CYCLONES,
    "anticyclone": CASE_ANTICYCLONES,
}

n_cases = max(len(CASE_CYCLONES), len(CASE_ANTICYCLONES))
fig, axes = plt.subplots(n_cases, 2, figsize=(14, 4 * n_cases), sharex=True)

for col_idx, pol in enumerate(("cyclone", "anticyclone")):
    ids = case_ids[pol]
    for row_idx in range(n_cases):
        ax = axes[row_idx, col_idx]
        if row_idx >= len(ids):
            ax.set_visible(False)
            continue

        tid = ids[row_idx]

        sub_pig = pig_medians[(pig_medians["track_id"] == tid) & (pig_medians["polarity"] == pol)]
        sub_pig = sub_pig.sort_values("date")
        ax.plot(sub_pig["date"], sub_pig["T chla"], "o-", color="black", ms=3, label="T chla")
        ax.set_ylabel("T chla")

        sub_pft = pft_medians[(pft_medians["track_id"] == tid) & (pft_medians["polarity"] == pol)]
        sub_pft = sub_pft.sort_values("date")
        ax2 = ax.twinx()
        top3 = sub_pft[PFT_COLS].mean().nlargest(3).index.tolist()
        for pft in top3:
            ax2.plot(sub_pft["date"], sub_pft[pft], "--", ms=2, lw=1, label=pft)
        ax2.set_ylabel("PFT fraction")

        ax.set_title(f"{pol} #{tid}", fontsize=10)
        if row_idx == 0:
            lines1, labels1 = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="upper right")

fig.autofmt_xdate()
fig.suptitle("Case study eddies - T chla + top PFTs over calendar time", y=1.01)
fig.tight_layout()